<a href="https://colab.research.google.com/github/internshipinabook/nlp-internship-in-a-book/blob/main/notebooks/week11_deploying_STARTER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Colab setup (skip if running locally) ──────────────────────
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/nlp-internship-in-a-book.git book
    %cd book
    !pip install -r requirements.txt -q
    print('Colab setup complete ✅')
else:
    print('Running locally ✅')


# Week 11 — Deploying a Text Classifier (STARTER)
### The NLP Internship | LinguaAI Labs

This week: model serialisation, FastAPI endpoint, Slack webhook, model card, monitoring.

> ⏸️ **Pause and Predict**
> Before writing the API endpoint: what happens if the model loads inside the endpoint function rather than at startup? (Hint: each API call is a request — think about how often the model would reload.)

In [ ]:
import sys, subprocess, importlib
for pkg in ["fastapi","uvicorn"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"])
import json, os, pickle
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings("ignore")
print("Setup complete ✅")
print("\nThis week requires the AfroXLMR model from Week 9.")
print("Ensure 'models/afro_xlmr/' exists, or run week9_fine_tuning_SOLUTION.ipynb first.")

## Part 1 — Serialise the Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import json, os

MODEL_DIR = "models/afro_xlmr_production"
os.makedirs(MODEL_DIR, exist_ok=True)

# YOUR CODE HERE — save the fine-tuned model and tokeniser
# afro_model.save_pretrained(MODEL_DIR)
# afro_tok.save_pretrained(MODEL_DIR)

# Save label mapping
label2id = {"billing":0,"technical":1,"delivery":2,"feedback":3,"abuse":4,"other":5}
id2label  = {v:k for k,v in label2id.items()}
with open(f"{MODEL_DIR}/label_map.json","w") as f:
    json.dump({"id2label":id2label,"label2id":label2id},f)

# Reload and verify
print("Verifying saved model...")
try:
    tok_loaded   = AutoTokenizer.from_pretrained(MODEL_DIR)
    model_loaded = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
    print(f"Model reloads successfully ✅")
    print(f"Parameters: {sum(p.numel() for p in model_loaded.parameters()):,}")
except Exception as e:
    print(f"Save model first — run Week 9 notebook: {e}")

## Part 2 — FastAPI Endpoint

Save the code below as `kova_classifier/app.py`.

> 🧠 **Debug This**
> The code at the bottom of this cell loads the model inside the endpoint function. This causes severe performance problems. Identify the bug and explain the fix.
>
> ```python
> @app.post('/classify')
> def classify_ticket(req: TicketRequest):
>     tokenizer = AutoTokenizer.from_pretrained('models/afro_xlmr_production')
>     model = AutoModelForSequenceClassification.from_pretrained('models/afro_xlmr_production')
>     ...
> ```
> **Bug:** model reloads on every request — 2-10 second latency per call
> **Fix:** load at module startup (outside the function) — loads once when the app starts

In [ ]:
# app.py — save this file as kova_classifier/app.py
APP_CODE = '''
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch, json, os

app = FastAPI(title="LinguaAI Ticket Classifier", version="1.0")

# Load ONCE at startup — not inside the endpoint
MODEL_DIR = "models/afro_xlmr_production"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

with open(f"{MODEL_DIR}/label_map.json") as f:
    maps = json.load(f)
ID2LABEL = {int(k): v for k, v in maps["id2label"].items()}

PIDGIN_MARKERS = {"abeg","una","don","dey","wahala","wetin","shey"}
ABUSE_THRESHOLD  = 0.80
REVIEW_THRESHOLD = 0.60

class TicketRequest(BaseModel):
    ticket_id: str
    text: str

@app.post("/classify")
def classify_ticket(req: TicketRequest):
    inputs = tokenizer(req.text, return_tensors="pt",
                       truncation=True, max_length=128)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs     = torch.softmax(logits, dim=-1)[0]
    pred_id   = probs.argmax().item()
    category  = ID2LABEL[pred_id]
    confidence= probs[pred_id].item()

    pidgin_heavy = len(set(req.text.lower().split()) & PIDGIN_MARKERS) >= 2
    abuse_id = next(k for k,v in ID2LABEL.items() if v=="abuse")

    return {
        "ticket_id": req.ticket_id,
        "category":  category,
        "confidence": round(confidence, 3),
        "abuse_alert": category=="abuse" and probs[abuse_id].item() >= ABUSE_THRESHOLD,
        "requires_human_review": confidence < REVIEW_THRESHOLD or pidgin_heavy,
    }

@app.get("/health")
def health(): return {"status": "healthy"}
'''
os.makedirs("kova_classifier", exist_ok=True)
with open("kova_classifier/app.py","w") as f: f.write(APP_CODE)
print("Saved: kova_classifier/app.py ✅")
print("\nTo run: uvicorn kova_classifier.app:app --reload --port 8000")
print("To test: curl -X POST http://localhost:8000/classify -H 'Content-Type: application/json'")
print('To test: POST /classify with {"ticket_id":"T001","text":"my account is locked"}')

## Part 3 — Slack Webhook Integration

In [ ]:
import os, requests

# Set via environment variable — NEVER hardcode
SLACK_WEBHOOK_URL = os.environ.get("SLACK_WEBHOOK_URL", "")

def post_to_slack(classification_result: dict) -> bool:
    """Post ticket classification to #ticket-triage Slack channel."""
    if not SLACK_WEBHOOK_URL or "YOUR/WEBHOOK" in SLACK_WEBHOOK_URL:
        print("⚠️  SLACK_WEBHOOK_URL not set — skipping. Set via:")
        print("    export SLACK_WEBHOOK_URL='https://hooks.slack.com/services/...'")
        return False

    category = classification_result["category"]
    EMOJI = {"billing":"💳","technical":"🔧","delivery":"📦",
              "feedback":"💬","abuse":"🚨","other":"📋"}

    message = {"blocks":[{"type":"section","text":{"type":"mrkdwn","text":
        f"{EMOJI.get(category,'📋')} *New ticket* | ID: {classification_result['ticket_id']} | "
        f"Category: {category} | Confidence: {classification_result['confidence']:.0%}"}}]}

    if classification_result.get("abuse_alert"):
        message["blocks"].insert(0,{"type":"header","text":{
            "type":"plain_text","text":"🚨 ABUSE ALERT — Immediate Review Required"}})

    resp = requests.post(SLACK_WEBHOOK_URL, json=message)
    return resp.status_code == 200

print("Slack integration ready.")
print("Set SLACK_WEBHOOK_URL environment variable to activate.")

## Part 4 — Model Card

In [ ]:
MODEL_CARD = """
# Model Card — LinguaAI CustomerCare Classifier v1.0

## Model Details
- Architecture: AfroXLMR-mini (fine-tuned)
- Task: 6-class support ticket classification
- Languages: English, Nigerian Pidgin, Yorùbá-English, Hausa-English
- Training data: 4,000 labelled tickets from CustomerCare-50K
- Training time: ~6 min GPU / ~35 min CPU

## Performance
| Metric | Value |
|--------|-------|
| Weighted F1 (overall) | [update] |
| Abuse recall | [update] |
| English F1 | [update] |
| Pidgin F1 | [update] |

## Fairness Findings
A [N]-point F1 gap exists between English and Pidgin tickets.
Pidgin-heavy tickets are routed to human review.
Retraining with 500 additional Pidgin labels committed within 60 days.

## Intended Use
Automated routing of incoming support tickets.
NOT for: autonomous account decisions, legal proceedings, unreviewed abuse escalation.

## Limitations
1. Trained on one quarter of ticket data — new complaint types may underperform
2. Abuse recall below 0.75 threshold — human review required for abuse-category predictions
3. Code-switched tickets with 3+ languages: lower reliability

## Monitoring
Weekly per-language F1 | Monthly abuse recall
Retrain trigger: Pidgin F1 < 0.55 OR abuse recall < 0.65

## Reference
Mitchell et al. (2019) Model Cards for Model Reporting. FAccT 2019.
"""

with open("outputs/model_card.md","w") as f: f.write(MODEL_CARD)
print("Model card saved: outputs/model_card.md ✅")
print(MODEL_CARD[:500])

## Part 5 — Monitoring Function

In [ ]:
import json
from datetime import datetime

def monitor_model_performance(sample_texts, pipeline_fn):
    """Weekly monitoring — classify a sample and flag drift."""
    import numpy as np
    results = [pipeline_fn(t) for t in sample_texts[:100]]
    confidences = [r.get("confidence",0) for r in results]
    abuse_rate  = sum(1 for r in results if r.get("category")=="abuse")/len(results)
    low_conf    = sum(1 for c in confidences if c < 0.60)/len(confidences)

    BASELINE = {"abuse_rate":0.08,"low_confidence_rate":0.12}
    alerts = []
    if abs(abuse_rate - BASELINE["abuse_rate"]) > 0.10:
        alerts.append(f"DRIFT: abuse_rate changed to {abuse_rate:.2f}")
    if low_conf > 0.25:
        alerts.append(f"DRIFT: low_confidence_rate={low_conf:.2f} > 0.25")

    report = {"date":datetime.now().isoformat(),"confidence_mean":np.mean(confidences),
              "low_confidence_rate":low_conf,"abuse_rate":abuse_rate,"alerts":alerts}
    print(f"Monitoring report: {report}")
    if alerts:
        print("⚠️  ALERTS:", alerts)
    else:
        print("✅ No significant drift detected")
    return report

print("Monitoring function defined.")
print("\nNote: monitoring requires the FastAPI app to be running (uvicorn) before calling.")
print("Run: uvicorn kova_classifier.app:app --port 8000")

## ⚠️ Common Mistakes

| Mistake | What goes wrong | Fix |
|---------|-----------------|-----|
| Deploying without a model card | A deployed model without documentation is a liability. Anyone using it has no idea what it does, what it does not do, or when to distrust it. | Model card must be complete before any deployment — not after |
| Saving only the model weights | Weights without the tokeniser, label encoder, and preprocessing pipeline cannot be loaded in production. Always save the complete inference pipeline. | Use joblib.dump or pickle to save the full pipeline as one object |

## 🏆 Challenge Task

> 🎯 **Core Path**
> Build a FastAPI or Streamlit app that accepts a ticket text and returns a predicted category with confidence score.

> 🔬 **Advanced Path**
> Add a confidence threshold to your app: if the model is less than 60% confident, return 'uncertain' instead of a category. Show the business case for this threshold.

## ✅ What You Can Do After This Week

- Serialise a fine-tuned Hugging Face model
- Build a FastAPI endpoint with fairness routing logic
- Connect a model API to Slack with abuse alerting
- Write a model card following Mitchell et al. (2019)
- Implement a production monitoring function

---
## ✅ Week 11 Complete
**Next:** `week12_capstone_STARTER.ipynb`

---
*Add the `kova_classifier/` folder to your internship portfolio.*

*The NLP Internship · LinguaAI Labs · github.com/InternshipInABook*

## ✅ By completing Week 11, you can now:

- Deploy a text classifier as a working FastAPI endpoint
- Save a complete inference pipeline — model, tokeniser, and label encoder together
- Write a model card that a deployment team could use without asking questions
- Implement a confidence threshold and explain the business rationale

📤 **GitHub:** Push week11_deploying_STARTER.ipynb and app.py. Commit: "Week 11: FastAPI endpoint live, model card complete"


---

## 📚 Get the Full Book

This notebook is part of **The NLP Internship** — Book 2 of the InternshipInABook™ Series.

The full book includes:
- Complete week-by-week narrative and explanations
- All STOP AND TRACE code walkthroughs
- Fairness briefs, model cards, and deployment guides
- Certificate of Completion

👉 **[Get the book on Selar →](https://selar.com/8440iqfr61)**

---
*InternshipInABook™ Series · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook)*
